# 10 · Геометрия камеры и двух видов: исследовательские стенды
Практикум к лекции 11. Девять стендов на CPU. Начальный рисунок появляется после выполнения ячейки стенда. Затем измените параметры и нажмите **«Вычислить»**. Задания расположены сразу под каждой панелью.

Выполните по три пункта каждого стенда. Для каждого опыта запишите прогноз, настройки до и после, числа или видимое изменение и объяснение. Когда сравниваете два значения, остальные настройки оставляйте одинаковыми.

## Координаты и смысл данных

`(u,v)` — пиксели, массив изображения индексируется `[v,u]`. Оси камеры: x вправо, y вниз, z вперед. `Xc = R @ Xworld + t`; R,t переводят **мир в камеру**. Центр камеры `C = -R.T @ t`. Проекция: `s*[u,v,1] = K @ Xc`. Нужна положительная глубина. Нет дисторсии; все сцены созданы кодом.

K: fx=fy=800 px, cx=320, cy=240; изображение 640×480. В эксперименте 160 точек имеют разные глубины 4…7.5 м, baseline .5 м, yaw камеры 2=.04rad, независимый Gaussian noise σ=.7px на каждую координату каждого кадра и 25%замененных target-наблюдений. GT-inlier означает, что соответствие не заменялось, а не что текущая модель согласуется с ним. Видимость обоих кадров проверяется до добавления шума.

Это неплоская 3D-сцена с известными камерами. Гомография плоского изображения из предыдущей работы не используется как датасет двух видов объемной сцены. Нет скачивания данных или весов.

In [ ]:
import importlib.util,subprocess,sys
requirements={'numpy':'numpy>=1.26,<3','cv2':'opencv-python-headless==4.12.0.88','matplotlib':'matplotlib>=3.8,<4','ipywidgets':'ipywidgets>=8,<9','threadpoolctl':'threadpoolctl>=3.5,<4'}
missing=[spec for module,spec in requirements.items() if importlib.util.find_spec(module) is None]
if missing:subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
import cv2
# В текущем Colab штатные варианты OpenCV конфликтуют: cv2 5 не содержит AKAZE.
# Исправляем только одноразовую среду Colab; локальную среду не меняем.
try:
    import google.colab as _colab_runtime
except ImportError:
    _colab_runtime=None
if _colab_runtime is not None and (cv2.__version__.split('.')[0]!='4' or not hasattr(cv2,'AKAZE_create')):
    import importlib.metadata as _opencv_metadata
    _variants=[]
    for _variant in ['opencv-python','opencv-python-headless','opencv-contrib-python','opencv-contrib-python-headless']:
        try:_opencv_metadata.version(_variant);_variants.append(_variant)
        except _opencv_metadata.PackageNotFoundError:pass
    if _variants:subprocess.check_call([sys.executable,'-m','pip','uninstall','-y',*_variants])
    subprocess.check_call([sys.executable,'-m','pip','install','-q','--no-deps','opencv-python-headless==4.12.0.88'])
    raise RuntimeError('OpenCV 4.12 установлен. В Colab выберите «Среда выполнения → Перезапустить сеанс», затем выполните ноутбук сверху.')
if cv2.__version__.split('.')[0]!='4':
    import importlib.metadata as meta
    installed=[]
    for variant in ['opencv-python','opencv-python-headless','opencv-contrib-python','opencv-contrib-python-headless']:
        try:meta.version(variant);installed.append(variant)
        except meta.PackageNotFoundError:pass
    if len(installed)==1:instruction=f'В отдельной ячейке выполните %pip install {installed[0]}==4.12.0.88 и перезапустите ядро; сохраняйте один вариант OpenCV.'
    else:instruction='Создайте чистую среду с одним вариантом opencv-python-headless==4.12.0.88 и выберите ее kernel.'
    raise RuntimeError('Для воспроизводимого протокола нужен OpenCV4.x. '+instruction)
from threadpoolctl import threadpool_limits
_thread_limit=threadpool_limits(limits=1)
try:
    import google.colab
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:pass
%matplotlib inline

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
import json,csv,time,platform,math
cv2.setNumThreads(1)
plt.rcParams.update({'figure.dpi':105,'font.size':10})
K0=np.array([[800.,0,320],[0,800.,240],[0,0,1.]])
SIZE=(640,480) # width,height

def skew(t):
    x,y,z=np.asarray(t).ravel();return np.array([[0,-z,y],[z,0,-x],[-y,x,0.]])

def camera(K,R,C):
    t=-R@np.asarray(C);return K@np.c_[R,t],t

def scene(seed=0,noise=.7,outlier_fraction=.25,baseline=.5,yaw=.04,kind='general',n=160):
    """Authored 3D synthetic observations. No detector, real photo or measured camera pose."""
    rng=np.random.default_rng(seed)
    X=rng.uniform([-1.,-.75,4.],[1.,.75,7.5],size=(n*5,3))
    if kind=='planar':X[:,2]=5.
    if kind=='pure_rotation':baseline=0.
    R,_=cv2.Rodrigues(np.array([0.,yaw,0.]));C=np.array([baseline,0.,0.])
    P1,t1=camera(K0,np.eye(3),np.zeros(3));P2,t2=camera(K0,R,C)
    p=cv2.projectPoints(X,np.zeros(3),t1,K0,None)[0][:,0]
    q=cv2.projectPoints(X,cv2.Rodrigues(R)[0],t2,K0,None)[0][:,0]
    visible=(p[:,0]>=0)&(p[:,0]<640)&(p[:,1]>=0)&(p[:,1]<480)&(q[:,0]>=0)&(q[:,0]<640)&(q[:,1]>=0)&(q[:,1]<480)
    X=X[visible][:n];p=p[visible][:n];q=q[visible][:n]
    if len(X)!=n:raise ValueError('Not enough jointly visible points for these cameras')
    clean1=p.copy();clean2=q.copy();p=p+rng.normal(0,noise,p.shape);q=q+rng.normal(0,noise,q.shape)
    inlier=np.ones(n,bool);bad=rng.choice(n,round(outlier_fraction*n),replace=False)
    q[bad]=rng.uniform([0,0],SIZE,(len(bad),2));inlier[bad]=False
    F=np.linalg.inv(K0).T@skew(t2)@R@np.linalg.inv(K0)
    return dict(X=X,p=p,q=q,clean1=clean1,clean2=clean2,inlier=inlier,R=R,t=t2,C=C,K=K0.copy(),P1=P1,P2=P2,F=F,noise=noise,kind=kind,baseline=baseline,seed=seed)

def native_sampson(F,p,q):
    if F is None or not np.isfinite(F).all() or np.linalg.norm(F)<1e-14:return np.full(len(p),np.inf)
    return np.sqrt(np.array([cv2.sampsonDistance(np.r_[a,1.].astype(float),np.r_[b,1.].astype(float),F) for a,b in zip(p,q)]))

def ray_angles(p,q,K,R):
    rays1=np.c_[p,np.ones(len(p))]@np.linalg.inv(K).T
    rays2=(np.c_[q,np.ones(len(q))]@np.linalg.inv(K).T)@R
    rays1/=np.linalg.norm(rays1,axis=1,keepdims=True);rays2/=np.linalg.norm(rays2,axis=1,keepdims=True)
    return np.degrees(np.arccos(np.clip(np.sum(rays1*rays2,axis=1),-1,1)))

def draw_scene(ax,s,selected=0):
    X=s['X'];ax.scatter(X[:,0],X[:,2],X[:,1],c=X[:,2],cmap='viridis',s=9,alpha=.65)
    for C,R,col,label in [(np.zeros(3),np.eye(3),'#2563eb','C1'),(s['C'],s['R'],'#ef4444','C2')]:
        corners=np.array([[0.,0,1],[640,0,1],[640,480,1],[0,480,1],[0,0,1]])
        rays=(corners@np.linalg.inv(s['K']).T*.6)@R+C
        ax.plot(rays[:,0],rays[:,2],rays[:,1],color=col)
        for Q in rays[:4]:ax.plot([C[0],Q[0]],[C[2],Q[2]],[C[1],Q[1]],color=col,lw=.8)
        Q=X[selected];ax.plot([C[0],Q[0]],[C[2],Q[2]],[C[1],Q[1]],color=col,ls='--',lw=1)
        ax.text(C[0],C[2],C[1],label,color=col)
    ax.set(xlabel='X (m)',ylabel='Z (m)',zlabel='Y down (m)',title='Known scene and viewing rays')
    from matplotlib.ticker import MaxNLocator
    for axis in (ax.xaxis,ax.yaxis,ax.zaxis):axis.set_major_locator(MaxNLocator(3))
    ax.tick_params(labelsize=7,pad=1)
    ax.invert_zaxis();ax.view_init(elev=20,azim=-65)

def image_axes(ax,title):
    ax.set(xlim=(0,640),ylim=(480,0),aspect='equal',xlabel='u (px)',ylabel='v (px)',title=title)

def draw_epiline(ax,line,color='#ef4444'):
    a,b,c=line;points=[]
    if abs(b)>1e-12:
        for u in [0,640]:
            v=-(a*u+c)/b
            if 0<=v<=480:points.append([u,v])
    if abs(a)>1e-12:
        for v in [0,480]:
            u=-(b*v+c)/a
            if 0<=u<=640:points.append([u,v])
    if len(points)>=2:points=np.array(points[:2]);ax.plot(points[:,0],points[:,1],color=color,lw=1.7)

def save_csv(path,rows):
    keys=list(dict.fromkeys(k for r in rows for k in r))
    with Path(path).open('w',newline='') as f:
        writer=csv.DictWriter(f,keys);writer.writeheader();writer.writerows(rows)
print('OpenCV',cv2.__version__,'; OpenCV threads',cv2.getNumThreads())

In [ ]:
import ipywidgets as W
from IPython.display import display
PANELS={}
from IPython.utils.capture import capture_output

def panel(name,fn,controls):
    out=W.Output();state={'calls':0,'last':None}
    def refresh(change=None):
        with capture_output(display=True) as captured:
            result=fn(**{key:control.value for key,control in controls.items()})
        items=[{'output_type':'display_data','data':item.data,'metadata':item.metadata}
               for item in captured.outputs]
        if captured.stdout:items.append({'output_type':'stream','name':'stdout','text':captured.stdout})
        if captured.stderr:items.append({'output_type':'stream','name':'stderr','text':captured.stderr})
        out.outputs=tuple(items);state['last']=result;state['calls']+=1
    for control in controls.values():
        if hasattr(control,'continuous_update'):control.continuous_update=False
        control.style.description_width = 'initial'
        control.layout.width = '285px'
    button=W.Button(description='Вычислить');button.on_click(refresh)
    PANELS[name]={'controls':controls,'state':state,'button':button,'refresh':refresh,'output':out}
    display(W.VBox([W.HBox(list(controls.values()),layout=W.Layout(flex_flow='row wrap')),button,out]));refresh()

def show_camera(focal,baseline,yaw,crop,update_k,depth,crop_x):
    s=scene(12,noise=0,outlier_fraction=0,baseline=baseline,yaw=np.radians(yaw),n=50)
    X=np.array([[1.,.5,depth]])
    K=K0.copy();K[0,0]=K[1,1]=focal
    s['K']=K.copy()
    R=s['R'];t=s['t'];cam=(R@X.T).T+t
    uv=cv2.projectPoints(X,cv2.Rodrigues(R)[0],t,K,None)[0][:,0]
    offset=np.array([crop_x,60.]) if crop else np.zeros(2)
    observed=uv[0]-offset;adjusted=K.copy()
    if update_k:adjusted[:2,2]-=offset
    predicted=cv2.projectPoints(X,cv2.Rodrigues(R)[0],t,adjusted,None)[0][0,0]
    error=float(np.linalg.norm(observed-predicted));w,h=(640-crop_x,360) if crop else SIZE
    fig=plt.figure(figsize=(11,4),layout='constrained');ax=fig.add_subplot(121,projection='3d');s['X']=np.vstack([X,s['X']]);draw_scene(ax,s)
    ax=fig.add_subplot(122);ax.scatter(*observed,s=90,c='#10b981',label='actual pixel after crop');ax.scatter(*predicted,s=140,marker='x',c='#ef4444',label='predicted with chosen K')
    ax.plot([observed[0],predicted[0]],[observed[1],predicted[1]],c='#ef4444');ax.set(xlim=(0,w),ylim=(h,0),aspect='equal',xlabel='u (px)',ylabel='v (px)',title=f'Projection error {error:.2f} px');ax.legend(loc='upper left')
    inside=bool(0<=observed[0]<w and 0<=observed[1]<h)
    if not inside:ax.text(.5,.5,f'Point outside frame\n(u, v) = ({observed[0]:.1f}, {observed[1]:.1f}) px',ha='center',transform=ax.transAxes)
    plt.show();plt.close(fig)
    print(f'C2={s["C"]}; t=-R C={t.round(3)}; camera XYZ={cam[0].round(3)}; normalized xy={(cam[0,:2]/cam[0,2]).round(4)}; pixel={observed.round(3)}. Crop offset={offset}.')
    return {'pixel':observed.tolist(),'predicted':predicted.tolist(),'error':error,'inside_image':inside,'camera_depth':float(cam[0,2]),'t':t.tolist()}

def show_epipolar(index,baseline,yaw,delta_v,delta_u,ray_depth,rise):
    s=scene(7,noise=0,outlier_fraction=0,baseline=baseline,yaw=np.radians(yaw),n=40)
    p=s['p'][index]
    C=np.array([baseline,rise,0.])
    _,t=camera(K0,s['R'],C)
    rvec=cv2.Rodrigues(s['R'])[0]
    q=cv2.projectPoints(s['X'][index:index+1],rvec,t,K0,None)[0][0,0]
    observed=q+[delta_u,delta_v]
    KI=np.linalg.inv(K0);F=KI.T@skew(t)@s['R']@KI;line=F@np.r_[p,1.];norm=np.linalg.norm(line[:2]);distance=None
    fig,axs=plt.subplots(1,2,figsize=(10,4),layout='constrained')
    for ax,pts,title in zip(axs,[s['p'],cv2.projectPoints(s['X'],rvec,t,K0,None)[0][:,0]],['Image 1: selected point','Image 2: possible epipolar line']):
        image_axes(ax,title);ax.scatter(pts[:,0],pts[:,1],c='#cbd5e1',s=15)
    axs[0].scatter(*p,c='#2563eb',s=90);axs[1].scatter(*q,c='#10b981',s=100,label='GT correspondence');axs[1].scatter(*observed,c='#f59e0b',s=70,marker='x',label='perturbed observation')
    if norm>1e-12:
        draw_epiline(axs[1],line);res=float(line@np.r_[observed,1.]);distance=abs(res)/norm
        nearest=observed-res*line[:2]/(norm*norm);axs[1].plot([observed[0],nearest[0]],[observed[1],nearest[1]],'--',color='#f59e0b')
        ray=np.linalg.inv(K0)@np.r_[p,1.]
        for depth,marker in [(4.,'s'),(8.,'^'),(ray_depth,'D')]:
            v=cv2.projectPoints((ray*depth).reshape(1,3),rvec,t,K0,None)[0][0,0]
            axs[1].scatter(*v,c='#64748b',s=38,marker=marker,label=f'Глубина луча {depth:g} м')
        axs[1].set_title(f'Point-to-line distance = {distance:.3f} px')
    else:axs[1].text(320,430,'Zero translation: F=0; no unique epipolar line',ha='center',fontsize=9)
    axs[1].legend(loc='upper right');plt.show();plt.close(fig)
    print('F maps a point to a line, not a point. Point-to-line distance here is not Sampson error. Pure rotation does not reveal depth.')
    return {'distance':distance,'zero_translation':baseline==0 and rise==0,'F_norm':float(np.linalg.norm(F))}

def show_robust(outliers,threshold,noise,budget,eval_px,seed,method):
    s=scene(seed,noise=noise,outlier_fraction=outliers);p,q=s['p'],s['q'];cv2.setRNGSeed(41)
    if method=='RANSAC':
        F,internal=cv2.findFundamentalMat(p,q,cv2.FM_RANSAC,threshold,.999,budget)
    else:
        F,internal=cv2.findFundamentalMat(p,q,cv2.FM_8POINT)
    error=native_sampson(F,p,q);accepted=error<=eval_px;tp=int((accepted&s['inlier']).sum());precision=tp/max(1,int(accepted.sum()));recall=tp/int(s['inlier'].sum()) if s['inlier'].any() else None
    fig,axs=plt.subplots(1,2,figsize=(11,4),layout='constrained')
    for i in range(50):
        color='#10b981' if accepted[i] and s['inlier'][i] else '#ef4444' if accepted[i] else '#cbd5e1'
        axs[0].plot([p[i,0],q[i,0]+640],[p[i,1],q[i,1]],c=color,lw=.7)
    axs[0].set(xlim=(0,1280),ylim=(480,0),aspect='equal',xlabel='image 1 | image 2 (u, px)',title=f'Первые 50 пар · проверка при {eval_px:g} px')
    axs[1].scatter(np.flatnonzero(s['inlier']),error[s['inlier']],s=12,c='#2563eb',label='GT inlier')
    axs[1].scatter(np.flatnonzero(~s['inlier']),error[~s['inlier']],s=18,c='#ef4444',marker='x',label='GT outlier');axs[1].set_yscale('symlog',linthresh=1);axs[1].set_ylim(bottom=0);axs[1].axhline(eval_px,c='k',ls='--')
    axs[1].set(xlabel='Correspondence index',ylabel='Sqrt Sampson error (px)',title=f'P={precision:.3f}, R={recall:.3f}' if recall is not None else 'No GT inliers: recall undefined');axs[1].legend();plt.show();plt.close(fig)
    print(f'Метод {method} · маска OpenCV: {int(internal.sum()) if internal is not None else 0} · по Sampson: {accepted.sum()}.')
    if method=='RANSAC':
        print(f'Подгонка: {threshold:g} px, до {budget} итераций. Внешний порог Sampson: {eval_px:g} px.')
    return {'precision':precision,'recall':recall,'accepted':int(accepted.sum()),'internal':int(internal.sum()) if internal is not None else 0,'median_gt_error':float(np.median(error[s['inlier']])) if np.isfinite(error).any() else None}

def show_triangulation(baseline,noise,flip_t,yaw,scale,min_angle,seed):
    s=scene(seed,noise=noise,outlier_fraction=0,baseline=baseline,yaw=np.radians(yaw),n=80)
    t=s['t']*scale*(-1 if flip_t else 1)
    angle=ray_angles(s['p'],s['q'],K0,s['R'])
    X,front=triangulated(s,s['R'],t)
    keep=front & (angle>=min_angle)
    error=np.abs(X[:,2]-s['X'][:,2])
    median=float(np.median(error[keep])) if keep.any() else None
    fig=plt.figure(figsize=(11,4),layout='constrained');ax=fig.add_subplot(121,projection='3d');draw_scene(ax,s)
    ax=fig.add_subplot(122);ax.plot([3,8],[3,8],'k--',label='ideal')
    if keep.any():
        colored=ax.scatter(s['X'][keep,2],X[keep,2],c=error[keep],cmap='magma',s=24)
        fig.colorbar(colored,ax=ax,shrink=.75,pad=.04,label='|Z - Zgt| (m); adaptive scale')
    ax.set(xlabel='GT depth (m)',ylabel='Estimated depth (m)',title=f'После проверок: {keep.sum()}/{len(keep)}')
    if median is None:ax.text(.5,.5,'Depth unavailable',transform=ax.transAxes,ha='center')
    ax.legend();plt.show();plt.close(fig)
    reprojection=reprojection_errors(s,X,s['R'],t,keep)
    reproj=float(np.median(reprojection)) if len(reprojection) else None
    print(f'Базис сцены: {baseline:g} м · базис модели: {baseline*scale:g} м · угол лучей (медиана): {np.median(angle):.3f}°')
    print(f'Перед обеими камерами: {front.sum()} · прошли порог угла: {keep.sum()} · |ΔZ|: {median} м · репроекция: {reproj} px (медианы)')
    return {'valid':int(keep.sum()),'median_depth_error_m':median,'median_angle_deg':float(np.median(angle)),'reprojection_px':reproj}


def triangulated(s,R,t):
    X=np.full_like(s['X'],np.nan)
    if np.linalg.norm(t)==0:
        return X,np.zeros(len(X),bool)
    P2=K0@np.c_[R,t]
    h=cv2.triangulatePoints(s['P1'],P2,s['p'].T,s['q'].T).T
    finite=np.abs(h[:,3])>1e-12
    X[finite]=h[finite,:3]/h[finite,3,None]
    front=finite & (X[:,2]>0) & ((X@R.T+t)[:,2]>0)
    return X,front


def project_known(X,R,t,K=K0):
    return cv2.projectPoints(X,cv2.Rodrigues(R)[0],t,K,None)[0][:,0]


def reprojection_errors(s,X,R,t,keep):
    if not keep.any():
        return np.empty(0)
    p=project_known(X[keep],np.eye(3),np.zeros(3))
    q=project_known(X[keep],R,t)
    return (np.linalg.norm(p-s['p'][keep],axis=1)+np.linalg.norm(q-s['q'][keep],axis=1))/2


def show_disparity(focal,baseline,depth,shift,step,delta,min_d):
    u1=320+focal*shift/depth
    u2=320+focal*(shift-baseline)/depth
    observed=u2+delta
    if step:
        observed=np.round(observed/step)*step
        u1=np.round(u1/step)*step
    d=u1-observed
    estimate=focal*baseline/d if baseline>0 and d>0 and d>=min_d else None
    fig,axs=plt.subplots(1,2,figsize=(11,3.8),layout='constrained')
    ax=axs[0]
    ax.scatter([u1,observed],[1,0],s=70)
    ax.plot([u1,observed],[1,0],ls='--')
    ax.axvline(u2,color='#64748b',alpha=.6,label='u₂ без ошибки и округления')
    ax.set(yticks=[0,1],yticklabels=['кадр 2','кадр 1'],ylim=(-.4,1.4),xlabel='u, px',title=f'Наблюдаемое смещение d = {d:.2f} px')
    ax.legend()
    ax=axs[1]
    d_grid=np.geomspace(.25,max(260,d*1.15),300)
    if baseline>0:
        ax.plot(d_grid,focal*baseline/d_grid,label='Z = fB / d')
    if estimate is not None:
        ax.scatter(d,estimate,color='#ef4444',label='оценка')
    ax.axhline(depth,color='#64748b',ls='--',label='глубина сцены')
    if min_d:
        ax.axvspan(.25,max(.25,min_d),color='#ef4444',alpha=.1)
    if baseline==0:
        ax.text(.5,.5,'Нулевой базис',transform=ax.transAxes,ha='center')
    ax.set(xscale='log',yscale='log',xlabel='d, px',ylabel='Z, м',title='Глубина из горизонтального смещения')
    ax.legend()
    plt.show()
    plt.close(fig)
    print('Глубина недоступна' if estimate is None else f'Глубина {estimate:.3f} м · ошибка {estimate-depth:+.3f} м')
    return {'d_px':float(d),'depth_m':None if estimate is None else float(estimate)}


def show_design(count,spread,kind,baseline,noise,units,seed):
    s=scene(seed,noise=noise,outlier_fraction=0,baseline=baseline,kind=kind,n=150)
    p,q=s['p'],s['q']
    near=np.argsort(np.linalg.norm(s['clean1'][:100]-K0[:2,2],axis=1))[:count]
    ids=np.arange(count) if spread=='Весь кадр' else near
    def coords(a):
        if units=='Координаты камеры':
            a=(a-K0[:2,2])/K0[0,0]
        return np.c_[a,np.ones(len(a))]
    A=np.einsum('ni,nj->nij',coords(q[ids]),coords(p[ids])).reshape(count,9)
    sv=np.linalg.svd(A,compute_uv=False)
    rank=int(np.linalg.matrix_rank(A))
    F,_=cv2.findFundamentalMat(p[ids],q[ids],cv2.FM_8POINT)
    train=np.zeros(len(p),bool)
    train[ids]=True
    hold=np.arange(len(p))>=100
    error=native_sampson(F,p,q)
    fit=float(np.median(error[train])) if np.isfinite(error).all() else None
    held=float(np.median(error[hold])) if np.isfinite(error).all() else None
    fig,axs=plt.subplots(1,2,figsize=(11,3.8),layout='constrained')
    image_axes(axs[0],'Выборка для подгонки F')
    axs[0].scatter(*p[hold].T,s=10,color='#cbd5e1',label='контроль')
    axs[0].scatter(*p[train].T,s=25,label='подгонка')
    axs[0].legend()
    axs[1].semilogy(np.arange(1,len(sv)+1),np.maximum(sv/sv[0],1e-18),'o-')
    axs[1].set(xticks=range(1,10),xlabel='Номер σ',ylabel='σᵢ / σ₁',title=f'Спектр матрицы A · численный ранг {rank}')
    plt.show()
    plt.close(fig)
    print(f'Медиана sqrt Sampson на подгонке / контроле: {fit} / {held} px. F оценивает OpenCV в пикселях при любом режиме показа A.')
    return {'rank':rank,'fit_px':fit,'held_px':held,'sigma8':float(sv[7]/sv[0])}


def show_pose(baseline,yaw,noise,outliers,count,min_angle,scale):
    s=scene(27,noise=noise,outlier_fraction=outliers,baseline=baseline,yaw=np.radians(yaw),n=count)
    fig,axs=plt.subplots(1,2,figsize=(11,3.8),layout='constrained')
    before,after=[],[]
    if baseline>0:
        R1,R2,t=cv2.decomposeEssentialMat(skew(s['t'])@s['R'])
        t=t.ravel()*scale
        for R,sign in [(R1,1),(R1,-1),(R2,1),(R2,-1)]:
            _,front=triangulated(s,R,sign*t)
            angle=ray_angles(s['p'],s['q'],K0,R)
            before.append(int(front.sum()))
            after.append(int((front & (angle>=min_angle)).sum()))
        ids=np.arange(4)
        axs[0].bar(ids-.18,before,.36,label='Z₁, Z₂ > 0')
        axs[0].bar(ids+.18,after,.36,label='и угол ≥ порога')
        axs[0].set(xticks=ids,xticklabels=['R₁ +t','R₁ −t','R₂ +t','R₂ −t'],ylim=(0,count*1.05),ylabel='Число пар')
        axs[0].legend()
        for R,sign,label in [(R1,1,'R₁ +t'),(R1,-1,'R₁ −t'),(R2,1,'R₂ +t'),(R2,-1,'R₂ −t')]:
            C=-R.T@(sign*t)
            axs[1].scatter(C[0],C[2],label=label)
            forward=R.T@np.array([0.,0.,1.])
            axs[1].quiver(C[0],C[2],forward[0],forward[2],angles='xy',scale_units='xy',scale=5/scale)
        axs[1].legend(loc='upper right',ncols=2)
    else:
        axs[0].text(.5,.5,'E = 0 · кандидаты не определены',ha='center',transform=axs[0].transAxes)
    axs[1].scatter(0,0,c='k',marker='+')
    axs[1].set(xlim=(-1.25*scale,1.25*scale),ylim=(-.6*scale,.7*scale),aspect='equal',xlabel='X, ед. модели',ylabel='Z, ед. модели',title='Кандидаты центров и оси камер (XZ)')
    plt.show()
    plt.close(fig)
    print(f'Голосование до и после порога угла: {before} → {after}. Длина t модели: {scale:g} при ненулевом базисе.')
    return {'front':before,'accepted':after}


def show_models(kind,baseline,yaw,noise,outliers,h_px,f_px):
    s=scene(62,noise=noise,outlier_fraction=outliers,baseline=baseline,yaw=np.radians(yaw),kind=kind)
    p,q=s['p'],s['q']
    H,_=cv2.findHomography(p,q,cv2.RANSAC,h_px)
    F,_=cv2.findFundamentalMat(p,q,cv2.FM_RANSAC,f_px,.999,3000)
    fig,axs=plt.subplots(1,2,figsize=(11,3.8),layout='constrained')
    image_axes(axs[0],'Куда H переносит точки первого кадра')
    axs[0].scatter(*s['clean2'].T,s=12,label='точная проекция')
    h_error=np.full(len(p),np.inf)
    if H is not None:
        predicted=cv2.perspectiveTransform(s['clean1'][None],H)[0]
        h_error=np.linalg.norm(predicted-s['clean2'],axis=1)
        axs[0].scatter(*predicted.T,s=14,marker='x',label='перенос по H')
        for a,b in zip(s['clean2'][::5],predicted[::5]):
            axs[0].plot([a[0],b[0]],[a[1],b[1]],color='#64748b',lw=.6)
    axs[0].legend()
    f_error=native_sampson(F,s['clean1'],s['clean2'])
    for values,label in [(h_error,'H: перенос точки'),(f_error,'F: sqrt Sampson')]:
        finite=np.isfinite(values)
        axs[1].scatter(s['X'][finite,2],values[finite],s=16,alpha=.65,label=label)
    axs[1].set(yscale='symlog',xlabel='Глубина сцены, м',ylabel='Остаток, px',title='Проверка на чистых парах сцены')
    axs[1].legend()
    plt.show()
    plt.close(fig)
    h_med=float(np.median(h_error)) if np.isfinite(h_error).all() else None
    f_med=float(np.median(f_error)) if np.isfinite(f_error).all() else None
    angle=float(np.median(ray_angles(s['clean1'],s['clean2'],K0,s['R'])))
    print(f'Медианы: H {h_med} px · F {f_med} px · угол лучей {angle:.4f}°. У H и F разные виды остатка.')
    return {'h_px':h_med,'f_px':f_med,'angle_deg':angle}


def show_pnp(count,noise,outliers,focal_error,pose_yaw,threshold,method):
    s=scene(15,noise=noise,outlier_fraction=outliers,yaw=np.radians(pose_yaw),n=120)
    X=s['X'][:count]
    q=s['q'][:count].copy()
    K=K0.copy()
    K[0,0]*=1+focal_error/100
    K[1,1]=K[0,0]
    cv2.setRNGSeed(15)
    if method=='RANSAC':
        ok,rvec,t,idx=cv2.solvePnPRansac(X,q,K,None,iterationsCount=500,reprojectionError=threshold,flags=cv2.SOLVEPNP_EPNP)
        used=0 if idx is None else len(idx)
    else:
        ok,rvec,t=cv2.solvePnP(X,q,K,None,flags=cv2.SOLVEPNP_EPNP)
        used=count
    fig,axs=plt.subplots(1,2,figsize=(11,3.8),layout='constrained')
    image_axes(axs[0],'2D-наблюдения для оценки позы')
    axs[0].scatter(*q.T,s=20,label='наблюдения')
    image_axes(axs[1],'Репроекция 3D-карты на контрольных точках')
    clean=s['clean2'][100:]
    axs[1].scatter(*clean.T,s=14,label='точная проекция')
    center=error=None
    if ok:
        R=cv2.Rodrigues(rvec)[0]
        t=t.ravel()
        axs[0].scatter(*project_known(X,R,t,K).T,s=20,marker='x',label='репроекция')
        predicted=project_known(s['X'][100:],R,t,K)
        axs[1].scatter(*predicted.T,s=20,marker='x',label='по оценённой позе')
        center=float(np.linalg.norm(-R.T@t-s['C']))
        error=float(np.median(np.linalg.norm(predicted-clean,axis=1)))
    else:
        axs[1].text(.5,.08,'Оценка позы недоступна',transform=axs[1].transAxes,ha='center')
    for ax in axs:
        ax.legend()
    plt.show()
    plt.close(fig)
    print(f'Позу удалось оценить: {ok} · участвовало в оценке: {used if ok else 0}/{count} · ошибка центра: {center} м · медиана на контроле: {error} px')
    return {'ok':bool(ok),'center_error_m':center,'held_px':error,'used':used if ok else 0}


def int_slider(label,value,low,high,step=1):
    return W.IntSlider(description=label,value=value,min=low,max=high,step=step)


def float_slider(label,value,low,high,step):
    return W.FloatSlider(description=label,value=value,min=low,max=high,step=step)


def choose(label,options):
    return W.Dropdown(description=label,options=options)


def toggle(label,value=False):
    return W.Checkbox(description=label,value=value,indent=False)

## Стенд 1. Проекция: положение камеры, фокус и crop

Слева сцена и два луча к выбранной точке. Справа наблюдаемый пиксель после обрезки и прогноз по выбранной матрице K. При обрезке начало координат сдвигается на указанное число пикселей слева и на 60 px сверху.

In [ ]:
panel('camera',show_camera,{
    'focal':int_slider('Фокус, px',800,400,1000,100),
    'baseline':float_slider('Базис, м',.5,0,1,.05),
    'yaw':int_slider('Поворот, °',0,-15,15),
    'depth':float_slider('Глубина точки, м',5,3,8,.5),
    'crop':toggle('Обрезка'),
    'crop_x':int_slider('Слева, px',80,0,160,20),
    'update_k':toggle('Обновить K',True),
})

### Задания после стенда 1

1. При нулевом повороте вручную вычислите пиксель для базиса 0,5 и 0,05 м. Сверьте координаты и знак t.
2. Включите обрезку, отключите обновление K. Предскажите норму ошибки при двух значениях обрезки слева, затем восстановите K. Запишите, какие параметры камеры изменились.
3. По очереди измените поворот, фокус и глубину. По напечатанным координатам камеры, нормализованным координатам и пикселю определите, на каком этапе действует каждый параметр.

## Стенд 2. Эпиполярная линия: куда может попасть точка

Известные камеры задают E и F. Линия во втором кадре получается как l₂ = F x₁. Ромб — проекция точки на заданной глубине первого луча. Отрезок от наблюдения до линии показывает измеряемое расстояние.

In [ ]:
panel('epipolar',show_epipolar,{
    'index':int_slider('Точка',0,0,39),
    'baseline':float_slider('Гориз. перенос, м',.5,0,1,.05),
    'rise':float_slider('Верт. перенос, м',0,-.6,.6,.05),
    'yaw':int_slider('Поворот, °',5,-15,15),
    'ray_depth':float_slider('Глубина ромба, м',5,3,10,.5),
    'delta_u':int_slider('Сдвиг u, px',0,-40,40,5),
    'delta_v':int_slider('Сдвиг v, px',0,-40,40,5),
})

### Задания после стенда 2

1. Для двух точек измените глубину ромба. Проверьте, изменились ли точка в первом кадре, линия и позиция ромба во втором.
2. Сравните сдвиг на 15 px по u и по v при горизонтальном переносе и нулевом повороте. Добавьте вертикальный перенос и повторите. Объясните результат через нормаль к линии.
3. Обнулите обе компоненты переноса, сохранив поворот. Запишите, что осталось определено в пикселях и какой результат исчез.

## Стенд 3. Ректифицированное стерео: почему дальняя глубина чувствительна к пикселю

Оси камер параллельны, а перенос направлен вдоль x. Стенд округляет измеренные u на выбранную сетку и показывает, где измеренная диспаратность попадает на график Z = fB/d.

In [ ]:
panel('disparity',show_disparity,{
    'focal':int_slider('Фокус, px',800,400,1000,100),
    'baseline':float_slider('Базис, м',.5,0,1,.05),
    'depth':float_slider('Глубина, м',5,4,60,1),
    'shift':float_slider('Координата X, м',.4,-1,1,.05),
    'step':choose('Шаг округления',[('Без округления',0),('0,25 px',.25),('1 px',1),('2 px',2)]),
    'delta':float_slider('Ошибка u₂, px',0,-3,3,.25),
    'min_d':float_slider('Минимум d, px',0,0,5,.5),
})

### Задания после стенда 3

1. Для 5 и 30 м вычислите глубину вручную до и после ошибки +1 px в правом кадре. Сравните абсолютные ошибки.
2. При глубине 30 м и базисе 0,05 м найдите два положения X, при которых округление до целых даёт разную диспаратность.
3. Найдите настройку, в которой включение порога d приводит к отказу. Запишите, что приобретается и что теряется при таком отборе.

## Стенд 4. Восьмиточечная оценка: какие ограничения дают пары

Каждая пара добавляет строку в систему A f = 0. Спектр показывает, насколько выражены независимые ограничения. Подгонку F выполняет OpenCV, а остатки измеряются отдельно на использованных парах и на фиксированном контроле.

In [ ]:
panel('design',show_design,{
    'count':int_slider('Пар для подгонки',12,8,80),
    'spread':choose('Покрытие',['Весь кадр','Около центра']),
    'kind':choose('Сцена',[('Объём','general'),('Плоскость','planar'),('Чистое вращение','pure_rotation')]),
    'baseline':float_slider('Базис, м',.5,.05,1,.05),
    'noise':float_slider('Шум, px',.2,0,2,.1),
    'units':choose('Координаты A',['Пиксели','Координаты камеры']),
    'seed':int_slider('Seed сцены',23,0,50),
})

### Задания после стенда 4

1. При нулевом шуме сравните спектр и ранг на объёмной сцене и на плоскости. Повторите с шумом. Что произошло с малыми σ?
2. При ненулевом шуме сравните 8 и 30 пар, затем для 8 пар смените покрытие. Запишите ошибки на подгонке и на контроле.
3. Переключите единицы A. Какие числа изменились и какие остались прежними? Объясните по тому, какие координаты получает решатель OpenCV.

## Стенд 5. Робастная оценка F: гипотезы, согласие и внешний порог

Стенд сравнивает RANSAC OpenCV с восьмиточечной оценкой по всем парам. Первый порог действует при подгонке RANSAC. Второй применяет sqrt Sampson к уже рассчитанной F. Ошибки на графике размечены по известным синтетическим соответствиям.

In [ ]:
panel('robust',show_robust,{
    'outliers':float_slider('Доля выбросов',.25,0,.65,.05),
    'threshold':float_slider('Порог подгонки, px',1,.5,4,.5),
    'noise':float_slider('Шум, px',.7,0,2,.1),
    'budget':choose('Лимит итераций',[('5000',5000),('20',20),('100',100),('500',500)]),
    'eval_px':float_slider('Порог оценки, px',2,.5,5,.5),
    'seed':int_slider('Seed сцены',44,0,50),
    'method':choose('Метод',[('RANSAC','RANSAC'),('F8 по всем парам','F8')]),
})

### Задания после стенда 5

1. При внешнем пороге 2 px сравните два метода на одинаковых парах. Запишите precision, recall и медиану ошибки настоящих соответствий.
2. При 55% выбросов сравните лимиты 20 и 5000. Повторите на втором seed. Объясните роль бюджета гипотез.
3. По очереди измените порог подгонки и внешний порог. Сверьте маску OpenCV, число принятых пар и остатки. Разберите, какая операция изменила модель.

## Стенд 6. Разложение E: четыре кандидата относительной позы

Из точной E стенд получает две матрицы R и два знака направления t. Каждая гипотеза триангулирует одни и те же наблюдения. График отделяет проверку положительных глубин от дополнительной проверки угла лучей.

In [ ]:
panel('pose',show_pose,{
    'baseline':float_slider('Базис сцены, м',.5,0,1,.05),
    'yaw':int_slider('Поворот, °',8,-15,15),
    'noise':float_slider('Шум, px',0,0,2,.2),
    'outliers':float_slider('Доля выбросов',0,0,.6,.05),
    'count':int_slider('Число пар',60,12,150,6),
    'min_angle':float_slider('Минимум угла, °',0,0,5,.2),
    'scale':float_slider('Длина t модели',1,.25,2,.25),
})

### Задания после стенда 6

1. Сопоставьте победителя по положительной глубине с центром и направлением камеры на схеме. Поменяйте поворот и повторите.
2. При малом ненулевом базисе включите шум, затем добавьте выбросы. Сравните голоса до и после порога угла; запишите покрытие победителя.
3. Измените длину t модели вдвое и сравните голоса при тех же изображениях. Затем обнулите базис сцены. Как отличаются произвольный масштаб и отсутствие переноса?

## Стенд 7. Триангуляция: репроекция, точность Z и доля принятых точек

Слева истинные камеры и сцена. Справа глубины триангулированных точек, прошедших обе проверки: положительное Z и минимальный угол лучей. Медиана ошибки репроекции измеряется на тех же точках.

In [ ]:
panel('triangulation',show_triangulation,{
    'baseline':float_slider('Базис сцены, м',.5,0,1,.05),
    'noise':float_slider('Шум, px',0,0,2,.2),
    'flip_t':toggle('Сменить знак t'),
    'yaw':int_slider('Поворот, °',2,-12,12),
    'scale':float_slider('Множитель базиса',1,.5,2,.25),
    'min_angle':float_slider('Минимум угла, °',0,0,5,.2),
    'seed':int_slider('Seed сцены',32,0,50),
})

### Задания после стенда 7

1. При шуме 1 px сравните базис 0,5 и 0,05 м. Запишите медиану угла и медианы ошибок глубины и репроекции.
2. При нулевом шуме смените знак t. Объясните причину отбраковки. Верните знак, удвойте базис модели и сравните ошибку глубины и репроекции.
3. Подберите порог угла, при котором часть точек отбрасывается. Приведите рядом медиану ошибки и долю оставшихся точек. Сравните с нулевым базисом.

## Стенд 8. Плоскость, чистое вращение и объёмная сцена: H и F

Оба решателя получают одинаковые зашумлённые пары. График проверяет их по точным проекциям этой сцены: H предсказывает вторую точку, а F ограничивает её положение линией. Это два разных геометрических остатка.

In [ ]:
panel('models',show_models,{
    'kind':choose('Сцена',[('Объём','general'),('Плоскость','planar'),('Чистое вращение','pure_rotation')]),
    'baseline':float_slider('Базис, м',.5,0,1,.05),
    'yaw':int_slider('Поворот, °',7,-15,15),
    'noise':float_slider('Шум, px',.3,0,2,.1),
    'outliers':float_slider('Доля выбросов',.15,0,.6,.05),
    'h_px':float_slider('Порог H, px',2,.5,5,.5),
    'f_px':float_slider('Порог F, px',2,.5,5,.5),
})

### Задания после стенда 8

1. Сравните перенос по H на плоскости и в объёмной сцене при тех же камерах. Найдите отрезки с заметной ошибкой и свяжите их с глубиной.
2. Включите чистое вращение, сравните ошибку переноса по H и угол лучей. Что можно восстановить без трансляционного параллакса?
3. Верните объёмную сцену и уменьшите базис. Зафиксируйте случай, где остаток F мал, а лучи почти параллельны. Объясните, почему остатки H и F нельзя читать как две оценки ошибки глубины.

## Стенд 9. PnP: добавить камеру к известной 3D-карте

Вход — точки 3D-карты и пиксели новой камеры. Стенд оценивает позу по части пар и проверяет её на фиксированных контрольных точках той же карты. Метрика центра выражена в метрах, поскольку координаты карты заданы в метрах.

In [ ]:
panel('pnp',show_pnp,{
    'count':int_slider('Пар для подгонки',40,8,100),
    'noise':float_slider('Шум, px',.7,0,2,.1),
    'outliers':float_slider('Доля выбросов',.25,0,.6,.05),
    'focal_error':int_slider('Ошибка фокуса, %',0,-25,25,5),
    'pose_yaw':int_slider('Поворот, °',8,-15,15),
    'threshold':float_slider('Порог RANSAC, px',3,.5,8,.5),
    'method':choose('Метод',['RANSAC','EPnP']),
})

### Задания после стенда 9

1. На одних данных сравните EPnP и PnP с RANSAC. Запишите число использованных пар, ошибку центра и остаток на контроле.
2. Уменьшите число пар и сравните два порога RANSAC при ненулевом шуме. Найдите настройку с потерей покрытия или отказом.
3. Введите ошибку фокуса и проверьте, что произошло с позой и контрольной репроекцией. Объясните, откуда берётся масштаб в PnP и где в SfM берётся сама карта.

## Источники

- [Hartley & Zisserman, Multiple View Geometry, глава 9](https://www.robots.ox.ac.uk/~vgg/hzbook/hzbook2/HZepipolar.pdf): `x2.T @ F @ x1=0`, F/E и эпиполярные линии.
- [Hartley, In Defense of the Eight-Point Algorithm](https://users.cecs.anu.edu.au/~hartley/Papers/fundamental/fundamental.pdf): нормализация координат для линейной оценки F.
- [OpenCV4.12: Camera Calibration and 3D Reconstruction](https://docs.opencv.org/4.12.0/d9/d0c/group__calib3d.html): projection, findFundamentalMat, recoverPose, triangulatePoints. `recoverPose` возвращает направление t с нормой 1; baseline в метрах из двух монокулярных кадров не определяется.
- [Fischler & Bolles, RANSAC, 1981](https://doi.org/10.1145/358669.358692): случайные гипотезы и согласованные наблюдения.

Все рисунки вычислены учебным кодом на авторской синтетике. Они не являются фотографиями, измеренным benchmark или воспроизведением полной SfM-системы.